# Improving reasoning — vector construction

Builds the reasoning control vector from **"Improving Reasoning Performance in Large Language Models via Representation Engineering"** ([arXiv:2504.19483](https://arxiv.org/abs/2504.19483)) on Mistral-7B-Instruct-v0.1.

Contrastive pairs (`reasoning_pairs.json`) share the same GSM8K-style questions, answered with sound vs. flawed reasoning; the difference of means of the last-token hidden states gives a control vector, exported as `reason.gguf` for `steer.ipynb`.


In [1]:
import json

with open("reasoning_pairs.json", encoding="utf-8") as f:
    pairs = json.load(f)

# Same question, sound vs. flawed reasoning, in Mistral [INST] format.
formatted_positive = [f"[INST] {p['question']} [/INST] {p['sound']}" for p in pairs]
formatted_negative = [f"[INST] {p['question']} [/INST] {p['flawed']}" for p in pairs]

In [2]:
import os

import easysteer.hidden_states as hs
from vllm import LLM
from vllm.steer_vectors.api import SelectSpec

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

MODEL = "/home/xhl/huggingface_models/mistralai/Mistral-7B-Instruct-v0.1"  # or mistralai/Mistral-7B-Instruct-v0.1

# Capture needs eager execution and prefix caching off: cache-hit
# tokens are never recomputed, so their hidden states can't be captured.
llm = LLM(
    model=MODEL,
    enforce_eager=True,
    enable_prefix_caching=False,
)

# Only the last prompt row feeds the extractor, so select it at the source.
result = hs.capture(
    llm,
    formatted_positive + formatted_negative,
    select=SelectSpec(phases=["prompt"], positions=[-1]),
)

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


INFO 08-04 00:52:36 [api_utils.py:273] non-default args: {'enable_prefix_caching': False, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/home/xhl/huggingface_models/mistralai/Mistral-7B-Instruct-v0.1'}


INFO 08-04 00:52:36 [model.py:623] Resolved architecture: MistralForCausalLM


INFO 08-04 00:52:36 [model.py:1788] Using max model len 32768


...

WARNING 08-04 00:55:39 [vllm.py:1249] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


(EngineCore pid=4003160) 

INFO 08-04 00:55:39 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


(EngineCore pid=4003160) 

INFO 08-04 00:55:39 [vllm.py:1428] Cudagraph is disabled under eager mode


In [3]:
from easysteer.steer import extract_diffmean_control_vector

# DiffMean over the last prompt token: mean(sound) − mean(flawed), per layer.
control_vector = extract_diffmean_control_vector(
    result,
    positive_indices=list(range(10)),
    model_type="llama",
    token_pos=-1,
    normalize=True,
)
control_vector.export_gguf("reason.gguf")

Computing DiffMean directions:   0%|          | 0/32 [00:00<?, ?it/s]

Computing DiffMean directions: 100%|██████████| 32/32 [00:00<00:00, 13279.68it/s]